# Example 3.6.2 / 1: Test a Marginal Prediction

Companion notebook for **3.6.2 LP Sensitivity Analysis** in the AMPL lab Documentation, section "Test a Marginal Prediction". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Test a Marginal Prediction

Use the production model from Duals and Reduced Costs (AMPL lab documentation, section 3.6). Save the base objective and
the `Time` dual, add `delta` hours, solve again, and compare prediction with observation. `value()` and `dual()`
read the base results:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set PROD;                                 # Products
param rate {PROD} > 0;                    # Production rate in tons per hour
param profit {PROD};                      # Profit per ton
param market {PROD} >= 0;                 # Maximum sales in tons
param avail >= 0;                         # Available production hours
var Make {p in PROD} >= 0, <= market[p];  # Tons to make, capped by sales
maximize Total_Profit: sum {p in PROD} profit[p] * Make[p];  # Total profit in dollars
# Hours used within avail; dual: value of an hour
subject to Time: sum {p in PROD} Make[p] / rate[p] <= avail;

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["PROD"] = ["bands", "coils"]
ampl.param["rate"] = {"bands": 200, "coils": 140}
ampl.param["profit"] = {"bands": 25, "coils": 30}
ampl.param["market"] = {"bands": 6000, "coils": 4000}
ampl.param["avail"] = 40

In [5]:
# run.py
ampl.solve(solver="highs", verbose=False)
z0 = ampl.obj["Total_Profit"].value()
y = ampl.con["Time"].dual()

for delta in [0.25, 30]:
    ampl.param["avail"] = 40 + delta
    ampl.solve(solver="highs", verbose=False)
    observed = ampl.obj["Total_Profit"].value() - z0
    print(delta, ampl.solve_result, "predicted", y * delta, "observed", observed)

0.25 solved predicted 1050.0 observed 1050.0
30 solved predicted 126000.0 observed 78000.0


### Pure AMPL: data.dat and model.run

Pure AMPL:

In [6]:
%%writefile data.dat
set PROD := bands coils;
param avail := 40;
param: rate profit market :=
  bands 200 25 6000
  coils 140 30 4000;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver highs;
option solver_msg 0;
solve;
param z0;  # Base-case profit in dollars
param y;   # Base-case dual of Time, dollars per hour
let z0 := Total_Profit;
let y := Time.dual;
for {delta in {0.25, 30}} {
  let avail := 40 + delta;
  solve;
  printf "%g %s predicted %g observed %g\n", delta, solve_result, y * delta, Total_Profit - z0;
}

0.25 solved predicted 1050 observed 1050
30 solved predicted 126000 observed 78000
